# 06 - ETF-Level Tuning

## Purpose of this notebook

In this notebook, I test the most flexible momentum model family in the project by allowing each ETF to use its own momentum lookback window.

Instead of using:
- one shared lookback for all ETFs
- or one lookback per group

I now allow each ETF to choose from a candidate set of momentum windows.

## Why this notebook matters

This is the most expressive version of the strategy so far. It has the highest potential to improve performance, but it also introduces the most model complexity and the highest overfitting risk.

The goal is to determine whether ETF-level customization improves the strategy enough to justify that added complexity.

## Main research question

Can ETF-level tuning outperform:
- the shared baseline model
- the manual asset-class-tuned model
- the correlation-cluster-tuned model

on a risk-adjusted and robust basis?

## Important evaluation principle

I will not choose the winner using only raw return.

Instead, I will compare models using:
- Validation Sharpe
- Test Sharpe
- Test Max Drawdown
- Test CAGR
- Validation Max Drawdown
- Full-sample Calmar

This helps separate genuinely strong models from models that simply maximize return at too much risk.

## 2. Import libraries

In this section, I import the libraries needed for:
- data handling
- plotting
- iterating through model combinations
- saving outputs

## What this section is doing

I use:
- `pandas` and `numpy` for time-series and calculations
- `matplotlib` for plots
- `itertools.product` to generate ETF-level momentum combinations
- `pathlib` for file paths

## What I can change later

- I can move stable helper functions into `src/`
- I can later parallelize the model search if needed

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import product

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True

## 3. Load processed data and earlier model outputs

In this section, I load:
- monthly returns
- monthly moving average filter
- monthly rolling volatility
- monthly momentum tables
- earlier model returns for comparison

## Why this matters

This lets me run the ETF-level backtest using the same base inputs as earlier notebooks, while also comparing the final result against the strongest earlier models.

In [3]:
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

monthly_returns = pd.read_csv(PROCESSED_DIR / "monthly_returns.csv", index_col=0, parse_dates=True)
ma_filter_monthly = pd.read_csv(PROCESSED_DIR / "ma_filter_monthly.csv", index_col=0, parse_dates=True)
rolling_vol_monthly = pd.read_csv(PROCESSED_DIR / "rolling_vol_monthly.csv", index_col=0, parse_dates=True)

momentum_tables = {
    60: pd.read_csv(PROCESSED_DIR / "momentum_60d_monthly.csv", index_col=0, parse_dates=True),
    120: pd.read_csv(PROCESSED_DIR / "momentum_120d_monthly.csv", index_col=0, parse_dates=True),
    180: pd.read_csv(PROCESSED_DIR / "momentum_180d_monthly.csv", index_col=0, parse_dates=True),
    252: pd.read_csv(PROCESSED_DIR / "momentum_252d_monthly.csv", index_col=0, parse_dates=True),
}

shared_baseline_returns = pd.read_csv(PROCESSED_DIR / "strategy_returns_shared_baseline.csv", index_col=0, parse_dates=True)
best_asset_class_returns = pd.read_csv(PROCESSED_DIR / "best_asset_class_returns.csv", index_col=0, parse_dates=True)
best_cluster_returns = pd.read_csv(PROCESSED_DIR / "best_correlation_cluster_returns.csv", index_col=0, parse_dates=True)

print("Monthly returns shape:", monthly_returns.shape)
print("MA filter shape:", ma_filter_monthly.shape)
print("Rolling vol shape:", rolling_vol_monthly.shape)

Monthly returns shape: (253, 8)
MA filter shape: (242, 8)
Rolling vol shape: (242, 8)


## 4. Define split, universe, and settings

In this section, I define:
- train / validation / test dates
- ETF universe
- top-K rule
- fallback asset
- candidate momentum windows

## Why this matters

These settings define the ETF-level tuning experiment.

In [4]:
TRAIN_START = "2005-01-01"
TRAIN_END = "2014-12-31"

VALID_START = "2015-01-01"
VALID_END = "2018-12-31"

TEST_START = "2019-01-01"
TEST_END = monthly_returns.index.max().strftime("%Y-%m-%d")

TOP_K = 2
FALLBACK_ASSET = "IEF"
MOMENTUM_WINDOWS = [60, 120, 180, 252]

ASSETS = ["SPY", "QQQ", "IWM", "EFA", "EEM", "TLT", "IEF", "GLD"]

print("Assets:", ASSETS)
print("Momentum windows:", MOMENTUM_WINDOWS)
print("Top K:", TOP_K)
print("Fallback asset:", FALLBACK_ASSET)

Assets: ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'IEF', 'GLD']
Momentum windows: [60, 120, 180, 252]
Top K: 2
Fallback asset: IEF


## 5. Align all inputs to a common monthly index

In this section, I align all return and signal tables to the same monthly dates.

## Why this matters

ETF-level tuning only makes sense if all signal inputs and returns line up exactly.

In [5]:
common_index = monthly_returns.index.intersection(ma_filter_monthly.index).intersection(rolling_vol_monthly.index)
for window in MOMENTUM_WINDOWS:
    common_index = common_index.intersection(momentum_tables[window].index)

common_index = common_index.sort_values()

monthly_returns = monthly_returns.loc[common_index, ASSETS]
ma_filter_monthly = ma_filter_monthly.loc[common_index, ASSETS]
rolling_vol_monthly = rolling_vol_monthly.loc[common_index, ASSETS]
for window in MOMENTUM_WINDOWS:
    momentum_tables[window] = momentum_tables[window].loc[common_index, ASSETS]

print("Common monthly observations:", len(common_index))
print("First date:", common_index.min())
print("Last date:", common_index.max())

Common monthly observations: 242
First date: 2006-01-31 00:00:00
Last date: 2026-02-27 00:00:00


## 6. Helper functions

In this section, I define helper functions for:
- inverse-volatility weighting
- performance metrics
- split metrics
- building an ETF-specific momentum table
- generating portfolio weights

## Why this matters

These functions keep the notebook organized and make the ETF-level search reproducible.

In [6]:
def inverse_vol_weights(vol_series):
    vol_series = vol_series.replace(0, np.nan).dropna()
    if len(vol_series) == 0:
        return pd.Series(dtype=float)
    inv_vol = 1 / vol_series
    return inv_vol / inv_vol.sum()


def compute_metrics(return_series):
    return_series = return_series.dropna()
    if len(return_series) == 0:
        return {}

    cumulative = (1 + return_series).cumprod()
    total_months = len(return_series)

    cagr = cumulative.iloc[-1] ** (12 / total_months) - 1
    ann_vol = return_series.std() * np.sqrt(12)
    sharpe = cagr / ann_vol if ann_vol != 0 else np.nan

    running_max = cumulative.cummax()
    drawdown = cumulative / running_max - 1
    max_dd = drawdown.min()

    calmar = cagr / abs(max_dd) if max_dd != 0 else np.nan

    return {
        "CAGR": cagr,
        "Annual Vol": ann_vol,
        "Sharpe": sharpe,
        "Max Drawdown": max_dd,
        "Calmar": calmar,
    }


def compute_split_metrics(return_series, train_start, train_end, valid_start, valid_end, test_start, test_end):
    train = return_series.loc[train_start:train_end]
    valid = return_series.loc[valid_start:valid_end]
    test = return_series.loc[test_start:test_end]

    train_metrics = compute_metrics(train)
    valid_metrics = compute_metrics(valid)
    test_metrics = compute_metrics(test)

    return {
        "Train Sharpe": train_metrics.get("Sharpe"),
        "Valid Sharpe": valid_metrics.get("Sharpe"),
        "Test Sharpe": test_metrics.get("Sharpe"),
        "Train CAGR": train_metrics.get("CAGR"),
        "Valid CAGR": valid_metrics.get("CAGR"),
        "Test CAGR": test_metrics.get("CAGR"),
        "Train MaxDD": train_metrics.get("Max Drawdown"),
        "Valid MaxDD": valid_metrics.get("Max Drawdown"),
        "Test MaxDD": test_metrics.get("Max Drawdown"),
    }


def build_etf_momentum_table(asset_window_map, momentum_tables):
    parts = []
    for asset, window in asset_window_map.items():
        parts.append(momentum_tables[window][[asset]])
    combined = pd.concat(parts, axis=1)
    combined = combined[ASSETS]
    return combined


def generate_weight_table(momentum_df, ma_filter_df, vol_df, top_k=2, fallback_asset="IEF"):
    assets = momentum_df.columns.tolist()
    weight_records = []

    for dt in momentum_df.index:
        momentum_today = momentum_df.loc[dt]
        ma_today = ma_filter_df.loc[dt]
        vol_today = vol_df.loc[dt]

        weights = pd.Series(0.0, index=assets)

        eligible = momentum_today[ma_today].dropna().sort_values(ascending=False)
        selected_assets = eligible.head(top_k).index.tolist()

        if len(selected_assets) > 0:
            selected_vol = vol_today.loc[selected_assets].dropna()
            selected_weights = inverse_vol_weights(selected_vol)

            for asset, w in selected_weights.items():
                weights[asset] = w

        weight_sum = weights.sum()

        if weight_sum < 1.0:
            weights[fallback_asset] += (1.0 - weight_sum)

        weight_records.append(weights)

    return pd.DataFrame(weight_records, index=momentum_df.index)

## 7. Model ranking rule

In this section, I make the strategy-selection rule explicit.

## Why this matters

ETF-level tuning creates many more candidate models than earlier notebooks. Because of that, it is especially important to be clear about how I decide which models are strongest.

I do not choose the winner using only the highest CAGR.

Instead, I rank models using the following order:

### Primary ranking metrics
1. Validation Sharpe
2. Test Sharpe
3. Test Max Drawdown
4. Test CAGR

### Secondary tie-breakers
5. Validation Max Drawdown
6. Full-sample Calmar

This ranking rule reflects the goal of building a strategy that is:
- profitable
- risk-aware
- robust
- believable out of sample

## Why CAGR alone is not enough

A strategy can have a high CAGR but still be weak if it:
- takes too much volatility
- suffers deeper drawdowns
- collapses out of sample
- only looks good because of a lucky regime

That is why Sharpe, volatility, and drawdown remain central to model selection.

## 8. ETF-level grid search

In this section, I test all ETF-level momentum assignments.

## What this section is doing

For each ETF, I choose one momentum window from:
- 60
- 120
- 180
- 252

Then I:
- build the ETF-specific momentum table
- apply the same MA filter
- select the top K
- weight with inverse volatility
- shift weights by one month
- compute realized returns
- evaluate the model

## What I can change later

- If this becomes too slow, I can parallelize it
- I can reduce the search space if I later decide to test only a smaller set of windows

In [8]:
results = {}
metrics_rows = []

all_combinations = list(product(MOMENTUM_WINDOWS, repeat=len(ASSETS)))
print("Total ETF-level combinations:", len(all_combinations))

Total ETF-level combinations: 65536


In [9]:
for idx, combo in enumerate(all_combinations):
    asset_window_map = dict(zip(ASSETS, combo))
    label = "_".join([f"{asset}_{window}" for asset, window in asset_window_map.items()])

    mixed_momentum = build_etf_momentum_table(asset_window_map, momentum_tables)

    weights = generate_weight_table(
        momentum_df=mixed_momentum,
        ma_filter_df=ma_filter_monthly[mixed_momentum.columns],
        vol_df=rolling_vol_monthly[mixed_momentum.columns],
        top_k=TOP_K,
        fallback_asset=FALLBACK_ASSET,
    )

    shifted_weights = weights.shift(1)
    returns = (shifted_weights * monthly_returns[mixed_momentum.columns]).sum(axis=1, min_count=1)

    results[label] = {
        "asset_window_map": asset_window_map,
        "weights": weights,
        "shifted_weights": shifted_weights,
        "returns": returns,
    }

    full_metrics = compute_metrics(returns)
    split_metrics = compute_split_metrics(
        returns,
        TRAIN_START, TRAIN_END,
        VALID_START, VALID_END,
        TEST_START, TEST_END
    )

    row = {
        "Model": label,
        **{f"{asset}_Window": window for asset, window in asset_window_map.items()},
        **full_metrics,
        **split_metrics,
    }
    metrics_rows.append(row)

    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1} / {len(all_combinations)} combinations")

Processed 5000 / 65536 combinations
Processed 10000 / 65536 combinations
Processed 15000 / 65536 combinations
Processed 20000 / 65536 combinations
Processed 25000 / 65536 combinations


## 9. Build ETF-level metrics table

In this section, I collect the performance metrics for all ETF-level models into one dataframe.

## Why this matters

This becomes the main comparison table for ETF-level tuning.

In [ ]:
etf_tuning_metrics = pd.DataFrame(metrics_rows).set_index("Model")
etf_tuning_metrics.head()

## 10. Build ranking table and inspect top models

In this section, I rank the ETF-level models using the agreed strategy-selection rule.

## What this section is doing

I sort the models by:
1. Validation Sharpe
2. Test Sharpe
3. Test Max Drawdown
4. Test CAGR
5. Validation Max Drawdown
6. Full-sample Calmar

I also keep the main metrics visible so I can judge whether higher CAGR is coming with lower Sharpe or higher risk.

In [ ]:
ranked_etf_models = etf_tuning_metrics.sort_values(
    by=[
        "Valid Sharpe",
        "Test Sharpe",
        "Test MaxDD",
        "Test CAGR",
        "Valid MaxDD",
        "Calmar",
    ],
    ascending=[False, False, False, False, False, False]
)

ranked_etf_models.head(20)

## 11. Create a tradeoff table for ranking intuition

In this section, I create a smaller table showing the key ranking metrics side by side.

## Why this matters

This lets me see whether a model with higher CAGR is sacrificing:
- Sharpe
- volatility
- drawdown

This is the table I will use to judge whether extra return is actually worth the extra risk.

In [ ]:
ranking_tradeoff_table = ranked_etf_models[
    [
        "CAGR",
        "Annual Vol",
        "Sharpe",
        "Max Drawdown",
        "Calmar",
        "Valid Sharpe",
        "Test Sharpe",
        "Valid CAGR",
        "Test CAGR",
        "Valid MaxDD",
        "Test MaxDD",
    ]
].head(20)

ranking_tradeoff_table

## 12. Inspect the best ETF-level model

In this section, I inspect:
- the best ETF-level model name
- the asset-specific momentum windows it selected
- the metrics row for the best model

In [ ]:
best_etf_model = ranked_etf_models.index[0]
best_etf_returns = results[best_etf_model]["returns"]
best_etf_weights = results[best_etf_model]["shifted_weights"]
best_etf_window_map = results[best_etf_model]["asset_window_map"]

print("Best ETF-level model:")
print(best_etf_model)
print("\nAsset-specific momentum windows:")
print(best_etf_window_map)

display(ranked_etf_models.loc[[best_etf_model]])

## 13. Compare the best ETF-level model against earlier winners

In this section, I compare the best ETF-level model against:
- shared baseline strategies
- best manual asset-class model
- best correlation-cluster model

## Why this matters

ETF-level tuning only deserves to continue if it actually improves over the earlier models.

In [ ]:
comparison_returns = pd.concat(
    [
        shared_baseline_returns,
        best_asset_class_returns.rename(columns={best_asset_class_returns.columns[0]: "Best_Manual_AssetClass"}),
        best_cluster_returns.rename(columns={best_cluster_returns.columns[0]: "Best_Correlation_Cluster"}),
        best_etf_returns.rename("Best_ETF_Level_Model"),
    ],
    axis=1
).dropna()

comparison_cumulative = (1 + comparison_returns).cumprod()
comparison_cumulative.plot(title="Shared vs Grouped vs ETF-Level Tuning")
plt.ylabel("Portfolio Value")
plt.show()

## 14. Inspect the best ETF-level model's weights

In this section, I inspect the shifted weight table of the best ETF-level model.

## Why this matters

This helps verify that the portfolio construction still behaves sensibly under the tuned ETF-specific settings.

In [ ]:
best_weights_check = best_etf_weights.copy()
best_weights_check["WeightSum"] = best_weights_check.sum(axis=1)
display(best_weights_check.head(12))

## 15. Save ETF-level tuning outputs

In this section, I save the key outputs from the ETF-level tuning experiment.

## What this section is doing

I save:
- the full ETF-level metrics table
- the ranked ETF-level table
- the ranking tradeoff table
- the best ETF-level returns
- the best ETF-level weights

In [ ]:
etf_tuning_metrics.to_csv(PROCESSED_DIR / "etf_level_tuning_metrics.csv")
ranked_etf_models.to_csv(PROCESSED_DIR / "etf_level_tuning_ranked.csv")
ranking_tradeoff_table.to_csv(PROCESSED_DIR / "etf_level_tuning_tradeoff_table.csv")
best_etf_returns.to_csv(PROCESSED_DIR / "best_etf_level_returns.csv")
best_etf_weights.to_csv(PROCESSED_DIR / "best_etf_level_weights.csv")

print("Saved ETF-level tuning outputs to:", PROCESSED_DIR.resolve())

## 16. Key takeaways from this notebook

At this stage, I have tested the most flexible model family in the project by allowing each ETF to use its own momentum lookback window.

## What I accomplished

- tested ETF-specific momentum assignments
- backtested the resulting portfolios
- ranked models using a risk-adjusted selection rule
- created a tradeoff table to compare CAGR against Sharpe, volatility, and drawdown
- compared the best ETF-level model against earlier winners

## What I learned

This notebook answers whether ETF-level tuning provides enough improvement to justify its added complexity and overfitting risk.

## What comes next

After this notebook, the next logical step is:
- compare all final model families in one notebook
- add turnover and transaction-cost sensitivity
- decide which model is strongest overall